# Задача 4: Решение СЛАУ методом Якоби
---

### Алгоритм (из лекции 3)

Метод Якоби — простая итерация. На каждой итерации $k$ новое приближение вычисляется **только через значения предыдущей итерации** $x^{(k-1)}$:

$$x_i^{(k)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j=1,\, j \neq i}^{n} a_{ij}\, x_j^{(k-1)} \right), \quad i = 1, \dots, n$$

**Начальное приближение:** $x^{(0)} = (0, 0, \dots, 0)$

**Критерий остановки:** $\|x^{(k)} - x^{(k-1)}\|_\infty < \varepsilon$

**Условие сходимости (Теорема 1, лекция 3):** диагональное преобладание

$$|a_{ii}| > \sum_{j=1,\, j \neq i}^{n} |a_{ij}|, \quad i = 1, \dots, n$$

In [ ]:
# ============================================================
# Проверка условия диагонального преобладания (Теорема 1)
# |a_ii| > sum_{j≠i} |a_ij|,  i = 1,...,n
# ============================================================

def check_diagonal_dominance(A):
    n = len(A)
    for i in range(n):
        diag = abs(A[i][i])
        row_sum = 0.0
        for j in range(n):
            if j != i:
                row_sum += abs(A[i][j])
        if diag <= row_sum:
            return False
    return True

In [ ]:
# ============================================================
# Метод Якоби (формула 3 из лекции)
# x_i^(k) = (1/a_ii) * [b_i - sum_{j≠i} a_ij * x_j^(k-1)]
# Все j ≠ i берутся из ПРЕДЫДУЩЕЙ итерации
# Начальное приближение: x^(0) = (0, ..., 0)
# ============================================================

def jacobi(A, b, eps=1e-6, max_iter=1000):
    n = len(A)

    if not check_diagonal_dominance(A):
        print("Предупреждение: нет диагонального преобладания, сходимость не гарантирована")

    x = [0.0] * n          # x^(0) = 0

    for k in range(1, max_iter + 1):
        x_new = [0.0] * n

        for i in range(n):
            s = 0.0
            for j in range(n):         # суммируем все j ≠ i
                if j != i:
                    s += A[i][j] * x[j]    # берём x^(k-1)
            x_new[i] = (b[i] - s) / A[i][i]

        # критерий остановки: ||x^(k) - x^(k-1)||_inf < eps
        diff = max(abs(x_new[i] - x[i]) for i in range(n))
        x = x_new

        if diff < eps:
            return x, k

    return x, max_iter

In [ ]:
# ---------- вспомогательные функции для вывода ----------

def print_vector(name, v):
    print(f"{name} = [{', '.join(f'{vi:.6f}' for vi in v)}]\n")

def mat_vec(A, x):
    n = len(A)
    result = [0.0] * n
    for i in range(n):
        for j in range(n):
            result[i] += A[i][j] * x[j]
    return result

---
## Тест 1 — пример из лекции (2×2)

$$A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}, \quad b = \begin{pmatrix} 1 \\ -1 \end{pmatrix}$$

Формулы итерации (из лекции):

$$x_1^{(k)} = \frac{1 - x_2^{(k-1)}}{2}, \qquad x_2^{(k)} = \frac{-1 - x_1^{(k-1)}}{2}$$

Точное решение: $x^* = (1,\; -1)$

In [ ]:
A2 = [
    [2, 1],
    [1, 2],
]
b2 = [1, -1]

x2, iters2 = jacobi(A2, b2, eps=1e-6)

print("=" * 50)
print("  МЕТОД ЯКОБИ — пример из лекции (2×2)")
print("=" * 50)
print(f"Итераций: {iters2}")
print()
print_vector("x (результат)", x2)
print_vector("x* (точное)  ", [1.0, -1.0])

Ax2 = mat_vec(A2, x2)
residual2 = max(abs(Ax2[i] - b2[i]) for i in range(len(b2)))
print(f"Невязка ||Ax - b||_inf = {residual2:.2e}")

---
## Тест 2 — матрица 4×4 с диагональным преобладанием

$$A = \begin{pmatrix} 10 & 1 & 1 & 1 \\ 1 & 10 & 1 & 1 \\ 1 & 1 & 10 & 1 \\ 1 & 1 & 1 & 10 \end{pmatrix}, \quad b = \begin{pmatrix} 13 \\ 13 \\ 13 \\ 13 \end{pmatrix}$$

Точное решение: $x^* = (1, 1, 1, 1)$

In [ ]:
A4 = [
    [10, 1, 1, 1],
    [1, 10, 1, 1],
    [1, 1, 10, 1],
    [1, 1, 1, 10],
]
b4 = [13, 13, 13, 13]

x4, iters4 = jacobi(A4, b4, eps=1e-6)

print("=" * 50)
print("  МЕТОД ЯКОБИ — матрица 4×4")
print("=" * 50)
print(f"Диагональное преобладание: {check_diagonal_dominance(A4)}")
print(f"Итераций: {iters4}")
print()
print_vector("x (результат)", x4)
print_vector("x* (точное)  ", [1.0, 1.0, 1.0, 1.0])

Ax4 = mat_vec(A4, x4)
residual4 = max(abs(Ax4[i] - b4[i]) for i in range(len(b4)))
print(f"Невязка ||Ax - b||_inf = {residual4:.2e}")

---
## Проверка через numpy

In [ ]:
import numpy as np

for A, b, x, name in [(A2, b2, x2, "2×2"), (A4, b4, x4, "4×4")]:
    x_np = np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float))
    diff = max(abs(x[i] - x_np[i]) for i in range(len(x)))
    print(f"{name}: макс. расхождение с numpy = {diff:.2e}")

---
## Конспект: как работает метод Якоби

### Идея (лекция 3)

Из $i$-го уравнения выражаем $x_i$, подставляя значения **предыдущей** итерации:

$$x_i^{(k)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j \neq i} a_{ij}\, x_j^{(k-1)} \right)$$

### Условие сходимости (Теорема 1)

Если матрица $A$ имеет **строгое диагональное преобладание**:

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}|, \quad \forall i$$

— то метод Якоби сходится при любом начальном приближении.

### Отличие от метода Зейделя

| Якоби | Зейдель |
|:-----:|:-------:|
| $x_j^{(k-1)}$ — только предыдущая итерация | $x_j^{(k)}$ — уже обновлённые значения на этой итерации |
| Требует хранить два вектора | Обновляет на месте |